# Períodos de calentamiento 

En los modelos que hemos creado hasta ahora, los pacientes comienzan a llegar cuando el servicio abre y todos se van cuando cierra.
...
Veamos un ejemplo. Esta es una versión ligeramente modificada de... (mayor duración, más corridas, cálculo de resultados por ensayo).

Supondremos que este es un sistema abierto las 24 horas — imaginemos que se trata de una función de triaje en un servicio de urgencias.

## Programación del modelo

:::{.callout-tip}
A lo largo del código, todo lo nuevo que se ha añadido irá seguido del comentario `##NEW`; así que fíjate en ello en los siguientes bloques de código.
:::

### La clase g

Primero agregamos un nuevo parámetro: la duración del período de calentamiento.

Aquí, la duración de la simulación se fijó en 2880 y el `warm_up_period` en 1440. En teoría, esto significa que 2/3 de la duración es tiempo de análisis y el 1/3 restante es tiempo de calentamiento. Si te preocupa que el sistema tarde más en alcanzar el estado estable de lo que esperabas, ¡el calentamiento puede ser incluso más largo que la recolección de resultados si lo deseas!


In [ ]:
#| label: g_class

#Clase para sparare globales parámetro valores.

class g:

    #Tiempos entre llegadas

    patient_inter = 5



    #Tiempos de actividad

    mean_n_consult_time = 6



    #Número de recursos

    number_of_nurses = 1



    #Simulation meta parámetros

    sim_duration = 2880

    warm_up_period = 1440 ##NEW - this will be in addition para the sim_duration

    number_of_runs = 100



:::{.callout-tip}

Si te resulta más fácil hacer seguimiento, también podrías definir el calentamiento así:


In [ ]:
#| label: g_class_alt

results_collection_period = 2880

warm_up_period = 1440

total_sim_duration = results_collection_period + warm_up_period



:::

### La clase Patient

Nuestra clase de pacientes no cambia.

### La clase Model

En la clase del modelo, el método `attend_clinic` cambia.

Consultamos el tiempo transcurrido de la simulación con el atributo `self.env.now`.

Luego, cada vez que un paciente asiste a la clínica y utiliza el recurso de enfermería, comprobamos si el tiempo actual de simulación es posterior al número de unidades de tiempo que hemos definido como calentamiento.


In [ ]:
#| label: attend_clinic_func

#Función generadora que representa la ruta para pacientes que asisten a la

#clínica.

def attend_clinic(self, patient):

    #Actividad de consulta de enfermería

    start_q_nurse = self.env.now



    with self.nurse.request() as req:

        yield req



        end_q_nurse = self.env.now



        patient.q_time_nurse = end_q_nurse - start_q_nurse



        ##NUEVO - espara verifica si el período de calentamienpara ha pasado antes de

        #agregar cualquier resultado

        if self.env.now > g.warm_up_period:

            self.results_df.at[patient.id, "Q Time Nurse"] = (

                patient.q_time_nurse

            )



        sampled_nurse_act_time = random.expovariate(1.0 /

                                                    g.mean_n_consult_time)



        yield self.env.timeout(sampled_nurse_act_time)



Por ejemplo, si el tiempo de simulación va por 840 y nuestro `warm_up` es..., entonces este bloque —que añade el tiempo en cola de este paciente a nuestros registros— no se ejecutará:


In [ ]:
#| label: warm_up_bypassed_code

self.results_df.at[patient.id, "Q Time Nurse"] = (

    patient.q_time_nurse

)



Sin embargo, si el tiempo de simulación es 1680, por ejemplo, sí se ejecutará.

#### El método `calculate_run_results`

Como ahora no contaremos al primer paciente, debemos eliminar la entrada ficticia del primer paciente que creamos al configurar el dataframe.


In [ ]:
#| label: dummy_remove

#Method para calculate and sparare results over the run

def calculate_run_results(self):

    self.results_df.drop([1], inplace=True) ##NEW



    self.mean_q_time_nurse = self.results_df["Q Time Nurse"].mean()



#### El método `run`

A continuación, debemos ajustar la duración de nuestro modelo para reflejar la combinación del período en el que queremos recolectar resultados y el período de calentamiento.


In [ ]:
#| label: single_run

#Méparado para ejecutar una sola corrida de la simulación

def run(self):

    #Iniciar los generadores DES

    self.env.process(self.generator_patient_arrivals())



    #Ejecutar por la duración especificada en la clase g

    ##NEW - we need para tell the simulation para run for the specified duration

    #+ el período de calentamienpara si aún queremos la duración indicada completa

    self.env.run(until=(g.sim_duration + g.warm_up_period))



    #Calcular resultados de la corrida

    self.calculate_run_results()



    #Print patient level results for this run

    print (f"Run Number {self.run_number}")

    print (self.results_df)



### La clase `Trial`

Nuestra clase de ensayos no cambia.

...

También vamos a pasar de imprimir los dataframes a nivel de paciente a devolverlos como salida.


In [ ]:
#Méparado para ejecutar una sola corrida de la simulación

def run(self):

    #Iniciar los generadores DES

    self.env.process(self.generator_patient_arrivals())



    #Ejecutar por la duración especificada en la clase g

    #We need para tell the simulation para run for the specified duration

    #+ el período de calentamienpara si aún queremos la duración indicada completa

    self.env.run(until=(g.sim_duration + g.warm_up_period))



    #Calcular resultados de la corrida

    self.calculate_run_results()



    #Devolver los resultados a nivel de paciente para esta corrida

    return (self.results_df) ##NEW



A continuación, modificamos el método `run_trial` de la clase `Trial` para devolver los dataframes a nivel de paciente y, además, calcular un promedio por corrida y una cifra promedio global para todos los ensayos.


In [ ]:
#| label: edited_results_method

#| eval: false

#Method para run a trial

def run_trial(self):

    #Ejecutar la simulación por el número de corridas especificado en la clase g.

    #Para cada corrida, creamos una nueva instancia de la clase Model y llamamos a su

    #méparado run, que pone parado en marcha. Una vez que la corrida ha

    #terminado, extraemos los resultados almacenados de la corrida y los guardamos con

    #el número de corrida en el dataframe de resultados del ensayo. También devolvemos los

    #dataframes compleparas a nivel de paciente.



    #Primero, crear una lista vacía para almacenar nuestros dataframes a nivel de paciente.

    results_dfs = []



    for run in range(g.number_of_runs):

        my_model = Model(run)

        patient_level_results = my_model.run()



        print( self.df_trial_results)

        #Primero registremos nuestro tiempo medio de espera para esta corrida

        self.df_trial_results.loc[run] = [my_model.mean_q_time_nurse]



        #Luego trabajemos en nuestros dataframes de resultados a nivel de paciente

        #Empezamos redondeando parado a 2 decimales

        patient_level_results = patient_level_results.round(2)

        #Agregar una nueva columna que registre la corrida

        patient_level_results['run'] = run

        #Ahora simplemente agregaremos espara a nuestra lista vacía (o, después de la primera

        #vez que iteremos, como un dataframe adicional en nuestra lista)

        results_dfs.append(patient_level_results)



    all_results_patient_level = pd.concat(results_dfs)



    #Espara calcula el atribupara self.mean_q_time_nurse_trial

    self.calculate_means_over_trial()



    #Una vez que el ensayo (es decir, paradas las corridas) haya finalizado, devolver los resultados

    return self.df_trial_results, all_results_patient_level, self.mean_q_time_nurse_trial



### El código completo actualizado


In [ ]:
#| label: code_full_warm_up

#| echo: true

#| eval: true



import simpy

import random

import pandas as pd



#Clase para sparare globales parámetro valores.

class g:

    #Tiempos entre llegadas

    patient_inter = 5



    #Tiempos de actividad

    mean_n_consult_time = 6



    #Número de recursos

    number_of_nurses = 1



    #Simulation meta parámetros

    sim_duration = 2880

    number_of_runs = 20

    warm_up_period = 1440 ##NEW - this will be in addition para the sim_duration



#Clase representing patients coming in para the clínica.

class Patient:

    def __init__(self, p_id):

        self.id = p_id

        self.q_time_nurse = 0



#Clase representing our model of the clínica.

class Model:

    #Construcparar

    def __init__(self, run_number):

        #Configurar el enpararno de SimPy

        self.env = simpy.Environment()



        #Configurar contadores para usar como IDs de entidad

        self.patient_counter = 0



        #Configurar recursos

        self.nurse = simpy.Resource(self.env, capacity=g.number_of_nurses)



        #Establecer el número de corrida a partir del valor recibido

        self.run_number = run_number



        #Set up DataFrame para sparare patient-level results

        self.results_df = pd.DataFrame()

        self.results_df["Patient ID"] = [1]

        self.results_df["Q Time Nurse"] = [0.0]

        self.results_df.set_index("Patient ID", inplace=True)



        #Set up attributes that will sparare mean queuing times across the run

        self.mean_q_time_nurse = 0



    #Generaparar function that represents the DES generaparar for patient entrantes

    def generator_patient_arrivals(self):

        while True:

            self.patient_counter += 1



            p = Patient(self.patient_counter)



            self.env.process(self.attend_clinic(p))



            sampled_inter = random.expovariate(1.0 / g.patient_inter)



            yield self.env.timeout(sampled_inter)



    #Función generadora que representa la ruta para pacientes que asisten a la

    #clínica.

    def attend_clinic(self, patient):

        #Actividad de consulta de enfermería

        start_q_nurse = self.env.now



        with self.nurse.request() as req:

            yield req



            end_q_nurse = self.env.now



            patient.q_time_nurse = end_q_nurse - start_q_nurse



            ##NUEVO - espara verifica si el período de calentamienpara ha pasado antes de

            #agregar cualquier resultado

            if self.env.now > g.warm_up_period:

                self.results_df.at[patient.id, "Q Time Nurse"] = (

                    patient.q_time_nurse

                )



            sampled_nurse_act_time = random.expovariate(1.0 /

                                                        g.mean_n_consult_time)



            yield self.env.timeout(sampled_nurse_act_time)



    #Method para calculate and sparare results over the run

    def calculate_run_results(self):

        ##NEW - as we now won't count the first patient, we need para remove

        #the dummy first patient result entry we created when we set up the

        #dataframe

        self.results_df.drop([1], inplace=True)



        self.mean_q_time_nurse = self.results_df["Q Time Nurse"].mean()



    #Méparado para ejecutar una sola corrida de la simulación

    def run(self):

        #Iniciar los generadores DES

        self.env.process(self.generator_patient_arrivals())



        #Ejecutar por la duración especificada en la clase g

        ##NEW - we need para tell the simulation para run for the specified duration

        #+ el período de calentamienpara si aún queremos la duración indicada completa

        self.env.run(until=(g.sim_duration + g.warm_up_period))



        #Calcular resultados de la corrida

        self.calculate_run_results()



        #Devolver los resultados a nivel de paciente para esta corrida

        return (self.results_df)



#Clase representing a Trial for our simulation

class Trial:

    #Construcparar

    def  __init__(self):

        self.df_trial_results = pd.DataFrame()

        self.df_trial_results["Run Number"] = [0]

        self.df_trial_results["Mean Q Time Nurse"] = [0.0]

        self.df_trial_results.set_index("Run Number", inplace=True)



    #Method para calculate and sparare means across runs in the trial

    def calculate_means_over_trial(self):

        self.mean_q_time_nurse_trial = (

            self.df_trial_results["Mean Q Time Nurse"].mean()

        )



    def run_trial(self):

        #Ejecutar la simulación por el número de corridas especificado en la clase g.

        #Para cada corrida, creamos una nueva instancia de la clase Model y llamamos a su

        #méparado run, que pone parado en marcha. Una vez que la corrida ha

        #terminado, extraemos los resultados almacenados de la corrida y los guardamos con

        #el número de corrida en el dataframe de resultados del ensayo. También devolvemos los

        #dataframes compleparas a nivel de paciente.



        #Primero, crear una lista vacía para almacenar nuestros dataframes a nivel de paciente.

        results_dfs = []



        for run in range(g.number_of_runs):

            my_model = Model(run)

            patient_level_results = my_model.run()



            print( self.df_trial_results)

            #Primero registremos nuestro tiempo medio de espera para esta corrida

            self.df_trial_results.loc[run] = [my_model.mean_q_time_nurse]



            #Luego trabajemos en nuestros dataframes de resultados a nivel de paciente

            #Empezamos redondeando parado a 2 decimales

            patient_level_results = patient_level_results.round(2)

            #Agregar una nueva columna que registre la corrida

            patient_level_results['run'] = run

            #Ahora simplemente agregaremos espara a nuestra lista vacía (o, después de la primera

            #vez que iteremos, como un dataframe adicional en nuestra lista)

            results_dfs.append(patient_level_results)



        all_results_patient_level = pd.concat(results_dfs)



        #Espara calcula el atribupara self.mean_q_time_nurse_trial

        self.calculate_means_over_trial()



        #Una vez que el ensayo (es decir, paradas las corridas) haya finalizado, devolver los resultados

        return self.df_trial_results, all_results_patient_level, self.mean_q_time_nurse_trial



    #Method para print trial results, including averages across runs

    def print_trial_results(self):

        print ("Trial Results")

        #NOTA: Por ahora omitimos las impresiones de los daparas a nivel de paciente

        #print (self.df_trial_results)



        print (f"Mean Q Nurse : {self.mean_q_time_nurse_trial:.1f} minutes")



#Crear una nueva instancia de Trial y ejecutarla

my_trial = Trial()

df_trial_results_warmup, all_results_patient_level_warmup, means_over_trial_warmup = my_trial.run_trial()


In [ ]:
#| label: code_full_no_warm_up

#| echo: false

#| eval: true



import simpy

import random

import pandas as pd



#Clase para almacenar parámetros globaleses. No creamos una instancia de esta

#clase: solo nos referimos al plano de la clase para acceder a los valores

#internos.

class g:

    patient_inter = 5

    mean_n_consult_time = 6

    number_of_nurses = 1

    sim_duration = 2880+1440

    number_of_runs = 20



#Clase representing patients coming in para the clínica.  Here, patients have

#dos atribuparas que llevan consigo: su ID y el tiempo que

#pasaron en la cola para la enfermera. El ID se pasa cuando se crea un nuevo paciente

#.

class Patient:

    def __init__(self, p_id):

        self.id = p_id

        self.q_time_nurse = 0



#Clase representing our model of the clínica.

class Model:

    #Construcparar para configurar el modelo para una corrida. Pasamos un número de corrida cuando

    #creamos un nuevo modelo.

    def __init__(self, run_number):

        #Crear un enpararno de SimPy en el que vivirá parado

        self.env = simpy.Environment()



        #Crear un contador de pacientes (que usaremos como ID del paciente)

        self.patient_counter = 0



        #Crear un recurso de SimPy para representar a una enfermera, que vivirá en el

        #enpararno creado arriba. La cantidad de este recurso que tenemos está

        #especificada por la capacidad, y obtenemos este valor de nuestra clase g.

        self.nurse = simpy.Resource(self.env, capacity=g.number_of_nurses)



        #Almacenar el número de corrida recibido

        self.run_number = run_number



        #Crear un nuevo DataFrame de Pandas que almacenará algunos resultados frente al

        #ID del paciente (que usaremos como índice).

        self.results_df = pd.DataFrame()

        self.results_df["Patient ID"] = [1]

        self.results_df["Q Time Nurse"] = [0.0]

        self.results_df["Time with Nurse"] = [0.0]

        self.results_df.set_index("Patient ID", inplace=True)



        #Crear un atribupara para almacenar el tiempo medio en cola para la enfermera

        #a lo largo de esta corrida del modelo

        self.mean_q_time_nurse = 0



    #Una función generadora que representa el generador DES para la llegada de pacientes

    #entrantes

    def generator_patient_arrivals(self):

        #Usamos un bucle infinipara aquí para seguir haciendo espara indefinidamente mientras

        #se ejecuta la simulación

        while True:

            #Incrementar el contador de pacientes en 1 (espara significa que nuestro primer paciente

            #tendrá un ID de 1)

            self.patient_counter += 1



            #Crear un nuevo paciente: una instancia de la clase Patient que

            #definimos arriba. Recuerda, pasamos el ID al crear un

            #paciente; así que aquí pasamos el contador de pacientes para usarlo como ID.

            p = Patient(self.patient_counter)



            #Indicar a SimPy que inicie la función generadora attend_clinic con

            #este paciente (la función generadora que modelará el

            #recorrido del paciente por el sistema)

            self.env.process(self.attend_clinic(p))



            #Muestrear aleaparariamente el tiempo hasta que llegue el siguiente paciente. Aquí,

            #muestreamos de una distribución exponencial (común para tiempos entre llegadas),

            #y pasamos un valor lambda de 1 / media. La media

            #del tiempo entre llegadas se almacena en la clase g.

            sampled_inter = random.expovariate(1.0 / g.patient_inter)



            #Congelar esta instancia de la función hasta que

            #transcurra el tiempo entre llegadas muestreado arriba. Nota: el tiempo en

            #SimPy avanza en "Unidades de Tiempo", que pueden representar cualquier cosa

            #que desees (solo asegúrate de ser consistente dentro del modelo)

            yield self.env.timeout(sampled_inter)



    #Una función generadora que representa la ruta de un paciente que pasa

    #through the clínica.  Here the pathway is extremely simple - a patient

    #llega, espera para ver a una enfermera y luego se va.

    #El objepara paciente se pasa a la función generadora para que podamos

    #extraer información de él / registrar información en él

    def attend_clinic(self, patient):

        #Registrar la hora en que el paciente comenzó a hacer fila para una enfermera

        start_q_nurse = self.env.now



        #Este código pide un recurso de enfermería y realiza parado lo siguiente

        #bloque de código con ese recurso de enfermería retenido (y por lo tanpara

        #no utilizable por otro paciente)

        with self.nurse.request() as req:

            #Congelar la función hasta que se pueda satisfacer la solicitud de una enfermera.

            #El paciente está actualmente en la cola.

            yield req



            #Cuando llegamos a esta parte del código, el control ha vuelpara a

            #la función generadora y, por lo tanpara, la solicitud de una enfermera ha

            #sido satisfecha. Ahora tenemos a la enfermera y hemos dejado de hacer fila, así que

            #podemos registrar la hora actual como la hora en que terminamos de hacer fila.

            end_q_nurse = self.env.now



            #Calcular el tiempo que este paciente estuvo en la cola para la enfermera y

            #registrarlo en el atribupara del paciente para ello.

            patient.q_time_nurse = end_q_nurse - start_q_nurse



            #Ahora muestreamos aleaparariamente el tiempo de este paciente con la enfermera.

            #Aquí usamos una distribución exponencial por simplicidad, pero

            #normalmente usarías una distribución log-normal para un modelo real

            #(volveremos a eso). Como al muestrear los tiempos entre llegadas,

            #paramamos la media de la clase g y pasamos 1 / media

            #como el valor lambda.

            sampled_nurse_act_time = random.expovariate(1.0 /

                                                        g.mean_n_consult_time)



            #Aquí almacenaremos el tiempo de cola para la enfermera y el tiempo muestreado

            #para estar con la enfermera en el DataFrame de resultados asociado al

            #ID de este paciente. En modelos del mundo real, es posible que no quieras

            #guardar los tiempos de actividad muestreados, pero como este es un

            #modelo simple, lo haremos aquí.

            #Usamos una propiedad útil de pandas llamada .at, que funciona un poco

            #como .loc. .at nos permite acceder (y por tanpara cambiar)

            #una celda específica en nuestro DataFrame proporcionando la fila y la columna.

            #Aquí especificamos la fila como el ID del paciente (el índice) y la

            #columna del valor que queremos actualizar para ese paciente.

            self.results_df.at[patient.id, "Q Time Nurse"] = (

                patient.q_time_nurse)

            self.results_df.at[patient.id, "Time with Nurse"] = (

                sampled_nurse_act_time)



            #Congelar esta función durante el tiempo de actividad muestreado

            #arriba. Este es el paciente pasando tiempo con la enfermera.

            yield self.env.timeout(sampled_nurse_act_time)



            #Cuando transcurra el tiempo anterior, la función generadora volverá

            #aquí. Como no hemos escripara nada más, la función

            #simplemente terminará. Este es un sumidero. Podríamos elegir agregar

            #algo aquí si quisiéramos registrar algo, p. ej., un contador

            #del número de pacientes que salieron, registrar algo sobre

            #los pacientes que salieron en un sumidero particular, etc.



    #Este méparado calcula resultados sobre una sola corrida. Aquí solo calculamos

    #una media, pero en modelos reales probablemente querrás calcular más.

    def calculate_run_results(self):

        #Tomar la media de los tiempos de cola para la enfermera entre los pacientes en

        #esta corrida del modelo.

        self.mean_q_time_nurse = self.results_df["Q Time Nurse"].mean()



    #El méparado run inicia los generadores de entidades DES, ejecuta la simulación,

    #y a su vez llama a lo que haga falta para generar resultados de la corrida

    def run(self):

        #Iniciar nuestros generadores de entidades DES que crean nuevos pacientes. Tenemos

        #solo uno en este modelo, pero tendríamos que hacer espara para cada uno si

        #tuviéramos múltiples generadores.

        self.env.process(self.generator_patient_arrivals())



        #Ejecutar el modelo por la duración especificada en la clase g

        self.env.run(until=g.sim_duration)



        #Ahora que la corrida de la simulación ha finalizado, llamar al méparado que calcula

        #los resultados de la corrida

        self.calculate_run_results()



        #Devolver los resultados a nivel de paciente para esta corrida

        return (self.results_df)



#Clase que representa un Trial para nuestra simulación: un lote de corridas de simulación.

class Trial:

    #El construcparar configura un dataframe de pandas que almacenará los

    #resultados clave de cada corrida (aquí solo el tiempo medio de cola para la enfermera)

    #contra el número de corrida, con el número de corrida como índice.

    def  __init__(self):

        self.df_trial_results = pd.DataFrame()

        self.df_trial_results["Run Number"] = [0]

        self.df_trial_results["Mean Q Time Nurse"] = [0.0]

        self.df_trial_results.set_index("Run Number", inplace=True)



    #Method para calculate and sparare means across runs in the trial

    def calculate_means_over_trial(self):

        self.mean_q_time_nurse_trial = (

            self.df_trial_results["Mean Q Time Nurse"].mean()

        )



    def run_trial(self):

        #Ejecutar la simulación por el número de corridas especificado en la clase g.

        #Para cada corrida, creamos una nueva instancia de la clase Model y llamamos a su

        #méparado run, que pone parado en marcha. Una vez que la corrida ha

        #terminado, extraemos los resultados almacenados de la corrida y los guardamos con

        #el número de corrida en el dataframe de resultados del ensayo. También devolvemos los

        #dataframes compleparas a nivel de paciente.



        #Primero, crear una lista vacía para almacenar nuestros dataframes a nivel de paciente.

        results_dfs = []



        for run in range(g.number_of_runs):

            my_model = Model(run)

            patient_level_results = my_model.run()



            print( self.df_trial_results)

            #Primero registremos nuestro tiempo medio de espera para esta corrida

            self.df_trial_results.loc[run] = [my_model.mean_q_time_nurse]



            #Luego trabajemos en nuestros dataframes de resultados a nivel de paciente

            #Empezamos redondeando parado a 2 decimales

            patient_level_results = patient_level_results.round(2)

            #Agregar una nueva columna que registre la corrida

            patient_level_results['run'] = run

            #Ahora simplemente agregaremos espara a nuestra lista vacía (o, después de la primera

            #vez que iteremos, como un dataframe adicional en nuestra lista)

            results_dfs.append(patient_level_results)



        all_results_patient_level = pd.concat(results_dfs)



        #Espara calcula el atribupara self.mean_q_time_nurse_trial

        self.calculate_means_over_trial()



        #Una vez que el ensayo (es decir, paradas las corridas) haya finalizado, devolver los resultados

        return self.df_trial_results, all_results_patient_level, self.mean_q_time_nurse_trial



    #Method para print trial results, including averages across runs

    def print_trial_results(self):

        print ("Trial Results")

        print (self.df_trial_results)



        print (f"Mean Q Nurse : {self.mean_q_time_nurse_trial:.1f} minutes")



#Crear una nueva instancia de Trial y ejecutarla

my_trial = Trial()

df_trial_results, all_results_patient_level, means_over_trial = my_trial.run_trial()



#### El método `run`

A continuación, debemos ajustar la duración de nuestro modelo para reflejar la combinación del período que queremos recolectar y el período de calentamiento.


In [ ]:
#| eval: true

#| label: results_p_level_head

all_results_patient_level.head()



##### Con calentamiento

Con el calentamiento, los IDs de los pacientes comienzan más tarde.


In [ ]:
#| eval: true

#| label: results_warmup_p_level_head

all_results_patient_level_warmup.head()



#### Resultados por corrida

##### Sin calentamiento


In [ ]:
#| eval: true

#| label: results_trial_level_head

df_trial_results.round(2).head()



##### Con calentamiento

Con el calentamiento, los IDs de los pacientes comienzan más tarde.


In [ ]:
#| eval: true

#| label: results_warmup_trial_level_head

df_trial_results_warmup.round(2).head()



#### Resultados globales

Sin calentamiento, nuestro tiempo de espera promedio global es


In [ ]:
#| eval: true

#| echo: false

f"{means_over_trial.round(2)} minutes"



Con calentamiento, nuestro tiempo de espera promedio global es


In [ ]:
#| eval: true

#| echo: false

f"{means_over_trial_warmup.round(2)} minutes"



En general, se puede ver que el tiempo de calentamiento puede tener un impacto muy significativo en nuestros tiempos de espera.

Veamos esto en una gráfica.

#### Resultados en el tiempo


In [ ]:
#| eval: true

#| echo: true



import plotly.express as px



df_trial_results = df_trial_results.reset_index()

df_trial_results['Warm Up'] = 'No Warm Up'



df_trial_results_warmup = df_trial_results_warmup.reset_index()

df_trial_results_warmup['Warm Up'] = 'With Warm Up'



fig = px.histogram(

    pd.concat([df_trial_results, df_trial_results_warmup]).round(2).reset_index(),

    x="Warm Up",

    color="Run Number", y="Mean Q Time Nurse",

    barmode='group',

    title='Average Queue Times per Run - With and Without Warmups')



fig.show()
